# Vector Store Setup & Abstract Embedding

This notebook walks through the core LangChain concepts needed to build a
semantic search index over the Arabidopsis abstract corpus.

1. LangChain `Document`?
2. Load `PaperRecord`s from MongoDB
3. Chunk `PaperRecord`s → `Document`s
4. Embeddings
5. Build Vector Store
6. Test Similarity Search
7. Test Persistence
8. Cross-Encoder Reranking

In [1]:
import logging
import os

from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO)

# Vectorstore files persist here — gitignored
PERSIST_DIR = ".data/vectorstore"
COLLECTION_NAME = "arabidopsis_abstracts"

### 1. LangChain `Document`

The `Document` is the fundamental unit in LangChain's retrieval system.
It has two fields:
- **`page_content`** — the raw text that gets embedded into a vector
- **`metadata`** — a plain dict of arbitrary fields (id, title, year, etc.)
  used for display and filtering, but *not* embedded

Everything downstream (embeddings, vector stores, retrievers) operates on
`Document` objects, so the rest of the pipeline is model/store-agnostic.

In [2]:
from langchain_core.documents import Document

# Create a Document manually to see its structure
example_doc = Document(
    page_content="Arabidopsis thaliana is a model plant for studying flowering time.",
    metadata={
        "pubmed_id": "12345678",
        "title": "Flowering time regulation in Arabidopsis",
        "year": 2020,
        "journal": "Plant Cell",
        "corpus_tag": "arabidopsis",
    },
)

print(f"page_content: {example_doc.page_content}")
print(f"metadata: {example_doc.metadata}")

page_content: Arabidopsis thaliana is a model plant for studying flowering time.
metadata: {'pubmed_id': '12345678', 'title': 'Flowering time regulation in Arabidopsis', 'year': 2020, 'journal': 'Plant Cell', 'corpus_tag': 'arabidopsis'}


### 2. Load `PaperRecord`s from MongoDB

We use `load_papers` from the `corpus` module to fetch records that were
stored in Phase 1 (the PubMed → MongoDB pipeline). Each record is a
validated `PaperRecord` Pydantic object.

In [3]:
from llm_knowledge_discovery.corpus import load_papers

records = load_papers(COLLECTION_NAME)
print(f"Loaded {len(records)} records")

# Inspect the first record
record = records[0]
print("\nFirst record:")
print(f"pubmed_id: {record.pubmed_id}")
print(f"title: {record.title}")
print(f"year: {record.year}")
print(f"abstract: {record.abstract[:200]}...")

INFO:llm_knowledge_discovery.corpus.storage:[storage.py] Loaded 2047 records for corpus 'arabidopsis_abstracts'


Loaded 2047 records

First record:
pubmed_id: 41811503
title: Cytokinin response factor 1 acts as a negative regulator of cytokinin-mediated developmental pathways in Arabidopsis.
year: 2026
abstract: CRF1 functions as a negative regulator of cytokinin signaling in Arabidopsis at least in part by antagonizing ARR12-dependent SHY2 transactivation, thereby influencing primary root growth, rosette dev...


### 3. Chunk `PaperRecord`s → `Document`s

"Chunking" means splitting text into the pieces we'll embed.
For this corpus, each abstract is 150–300 words — a natural, self-contained
unit of meaning. So our default strategy is `whole_abstract`: one Document
per paper, with the full abstract as `page_content`.

The `strategy` parameter lets us swap in sentence-level chunking later
without changing anything downstream.

In [4]:
from llm_knowledge_discovery.vectorstore import chunk_records

docs = chunk_records(records, strategy="whole_abstract")
print(f"{len(docs)} documents created")

# Show the first Document — note page_content = abstract text
doc = docs[0]
print("\nFirst document:")
print(f"page_content: {doc.page_content[:200]}...")
print(f"metadata: {doc.metadata}")

/Users/dylanelliott/workspace/llm-knowledge-discovery/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:llm_knowledge_discovery.vectorstore.chunking:[chunking.py] Chunked 2047 records into 2047 documents using strategy='whole_abstract'


2047 documents created

First document:
page_content: CRF1 functions as a negative regulator of cytokinin signaling in Arabidopsis at least in part by antagonizing ARR12-dependent SHY2 transactivation, thereby influencing primary root growth, rosette dev...
metadata: {'pubmed_id': '41811503', 'title': 'Cytokinin response factor 1 acts as a negative regulator of cytokinin-mediated developmental pathways in Arabidopsis.', 'year': 2026, 'journal': 'Planta', 'collection_name': 'arabidopsis_abstracts'}


### 4. Embeddings

We use `HuggingFaceEmbeddings` with the `all-MiniLM-L6-v2` model — a fast,
lightweight sentence encoder that runs entirely locally (no API key needed).
It produces 384-dimensional vectors. The vector store stores these vectors and 
uses them for nearest-neighbor search.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

# Loading the model downloads weights on first run (~90MB), then caches locally
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# embed_query converts a single string to a vector
vector = embeddings.embed_query("flowering time regulation")

print(f"Vector dimensions: {len(vector)}")
print(f"First 10 values: {[round(v, 4) for v in vector[:10]]}")

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu


Vector dimensions: 384
First 10 values: [-0.056, 0.0148, 0.0379, 0.0002, 0.0785, 0.099, 0.0244, -0.0578, 0.0514, 0.005]


### 5. Build Vector Store

**Chroma** is an embedded vector database — it runs in-process with no
extra server. We pass our Documents and an embedding function, and Chroma:
1. Calls `embeddings.embed_documents()` on each `page_content`
2. Stores the resulting vectors alongside the original text + metadata
3. Builds an HNSW index for fast approximate nearest-neighbor search
4. Persists everything to `.data/vectorstore/` so we don't re-embed on restart

This step may take a few minutes the first time (embedding ~1000+ abstracts).

In [6]:
from llm_knowledge_discovery.vectorstore import build_vectorstore

vectorstore = build_vectorstore(
    documents=docs,
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

print(f"Vector store built at: {PERSIST_DIR}")
print(f"Files created: {os.listdir(PERSIST_DIR)}")

INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Building vectorstore: collection='arabidopsis_abstracts', model='all-MiniLM-L6-v2', docs=2047, persist_dir='.data/vectorstore'
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Deleted existing collection 'arabidopsis_abstracts'
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Vectorstore built and persisted to '.data/vectorstore'


Vector store built at: .data/vectorstore
Files created: ['bead055b-7c53-4434-97f2-393f186f197d', 'chroma.sqlite3', '9336c6dd-6e1a-4a3d-a8eb-492bfa2a2cd5', '7f13fe4b-771b-4285-8af2-7df1ab2eb10f']


### 6. Test Similarity Search

The titles returned should be semantically relevant to the query — this
confirms the embedding + indexing pipeline is working end-to-end.

In [ ]:
query = "flowering time regulation"
results = vectorstore.similarity_search(query, k=10)

print(f"Top 10 results for: '{query}'\n")
for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.metadata['title']}")
    print(f"{doc.page_content[:150]}...")
    print()

Top 3 results for: 'flowering time regulation'

[1] BRASSINAZOLE RESISTANT 1 delays photoperiodic flowering by repressing CONSTANS transcription.
Photoperiodic regulation of flowering time plays a critical role in plant reproductive success and crop yield. In Arabidopsis thaliana, the expression...

[2] The B-box protein BBX13/COL15 suppresses photoperiodic flowering by attenuating the action of CONSTANS in Arabidopsis.
The optimal timing of transition from vegetative to floral reproductive phase is critical for plant productivity and agricultural yields. Light plays ...

[3] CONSTANS alters the circadian clock in Arabidopsis thaliana.
Plants are sessile organisms that have acquired highly plastic developmental strategies to adapt to the environment. Among these processes, the floral...



### 7. Test Persistence

Test that vector store persists across sessions. Load the Chroma store from 
disk (no re-embedding) and runs same query — should return identical results.

In [6]:
from llm_knowledge_discovery.vectorstore import load_vectorstore

# Load from disk — no re-embedding
vs_loaded = load_vectorstore(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

query = "flowering time regulation"
results_loaded = vs_loaded.similarity_search(query, k=10)

print(f"Loaded from disk. Top 10 results for: '{query}'\n")
for i, doc in enumerate(results_loaded):
    print(f"[{i+1}] {doc.metadata['title']}")
    print(f"{doc.page_content[:150]}...")
    print()

INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Loading vectorstore: collection='arabidopsis_abstracts', persist_dir='.data/vectorstore'
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Vectorstore loaded from '.data/vectorstore'


Loaded from disk. Top 10 results for: 'flowering time regulation'

[1] BRASSINAZOLE RESISTANT 1 delays photoperiodic flowering by repressing CONSTANS transcription.
Photoperiodic regulation of flowering time plays a critical role in plant reproductive success and crop yield. In Arabidopsis thaliana, the expression...

[2] The B-box protein BBX13/COL15 suppresses photoperiodic flowering by attenuating the action of CONSTANS in Arabidopsis.
The optimal timing of transition from vegetative to floral reproductive phase is critical for plant productivity and agricultural yields. Light plays ...

[3] CONSTANS alters the circadian clock in Arabidopsis thaliana.
Plants are sessile organisms that have acquired highly plastic developmental strategies to adapt to the environment. Among these processes, the floral...

[4] GI as a dynamic integrator: Synchronizing photoperiod and temperature signals to control flowering time in Arabidopsis.
GIGANTEA (GI) integrates photoperiod and temperature signa

### 8. Cross-Encoder Reranking

Use a cross-encoder model to re-rank the document similarity hist based on 
their predicted relevance to the query. The cross-encoder model takes both a 
document and the query and predicts a score of how relevant this document is 
for the query. 

In [7]:
from llm_knowledge_discovery.vectorstore import (
    load_vectorstore, rerank_documents
)

# Load from disk — no re-embedding
vectorstore = load_vectorstore(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

query = "flowering time regulation"

# Fetch a broad candidate set via fast bi-encoder similarity search
results = vectorstore.similarity_search(query, k=10)

# Re-rank the candidates with a cross-encoder for higher-quality ordering
reranked_results = rerank_documents(query, results, top_k=5)

print(f"Top 5 re-ranked results for: '{query}'\n")
for i, doc in enumerate(reranked_results):
    print(f"[{i+1}] {doc.metadata['title']}")
    print(f"{doc.page_content[:150]}...")
    print()

INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Loading vectorstore: collection='arabidopsis_abstracts', persist_dir='.data/vectorstore'
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Vectorstore loaded from '.data/vectorstore'
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents


Top 5 re-ranked results for: 'flowering time regulation'

[1] BRASSINAZOLE RESISTANT 1 delays photoperiodic flowering by repressing CONSTANS transcription.
Photoperiodic regulation of flowering time plays a critical role in plant reproductive success and crop yield. In Arabidopsis thaliana, the expression...

[2] Comparative transcriptome analysis reveals candidate gene for flowering time QTL HvHeading in barley.
Identifying genes regulating flowering time enhances understanding mechanisms that improve crop adaptation and productivity. This study aims to identi...

[3] Epigenetic regulation of floral transition: pathways and players.
The epigenetic mechanisms that regulate the DNA-histone contacts and the chromatin-based control of transcription provide an essential link between va...

[4] [Molecular regulatory mechanisms of TCP transcription factors and their roles in regulating flowering].
TCP transcription factors are a class of plant-specific regulators that play pivotal roles in p